# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [1]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant matplotlib seaborn

  Using cached mlcroissant-1.1.1-py2.py3-none-any.whl.metadata (11 kB)
Using cached mlcroissant-1.1.1-py2.py3-none-any.whl (166 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 14.8 MB/s  0:00:00eta 0:00:01
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.1
    Uninstalling matplotlib-3.10.1:
      Successfully uninstalled matplotlib-3.10.1 0/2 [matplotlib]
  Attempting uninstall: mlcroissant━━━━━━━━━━━━━ 0/2 [matplotlib]
    Found existing installation: mlcroissant 1.0.2232m0/2 [matplotlib]
    Uninstalling mlcroissant-1.0.22:━━━━━━━━ 0/2 [matplotlib]
      Successfully uninstalled mlcroissant-1.0.220/2 [matplotlib]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mlcroissant] [mlcroissant]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
croissant-duckdb 0.1.0 requires pandas<4,>=3.0.6, but you have pandas 2.2.3 w

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [2]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

import sys
import tempfile
import urllib.request

PATCH_URL = 'https://raw.githubusercontent.com/senscience/notebooks/main/tools/patch_fair2.py'
patch_folder = tempfile.mkdtemp()
urllib.request.urlretrieve(PATCH_URL, f'{patch_folder}/patch_fair2.py')
sys.path.insert(0, patch_folder)
from patch_fair2 import load_dataset

# Load the dataset metadata
dataset = load_dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {getattr(metadata,'cite_as', getattr(metadata,'citeAs', None))}")

mlcroissant can't read the fair2.json as published (ValueError): patching it
Name: Marianna_2026_ExtracellularVesicleMassSpectrometryDataset

Description: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

Cite as: Kulka, M., Rodriguez Miera, S., Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.


## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities are referenced by their `@id` fields as per the dataset Croissant schema.

In [3]:
# List all available record sets by @id
print('Available Record Sets:')
record_sets = getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))
if not record_sets:
    print('No RecordSets found in the metadata. Inspect metadata for available entities.')
else:
    for rset in record_sets:
        print(f"- ID: {rset.id}")
        print(f"  Name: {rset.name or 'N/A'}")

    # Show fields (columns) for each record set
    print('\nRecord Sets and Fields:')
    for rset in record_sets:
        print(f"- RecordSet @id: {rset.id}")
        if rset.fields:
            for field in rset.fields:
                print(f"    - Field @id: {field.id}, name: {field.name or 'N/A'}")
        else:
            print("    No fields found.")

Available Record Sets:
- ID: d18d681b-431c-4fc3-b535-1e26cb261034
  Name: Protein Abundance Analysis - Homo sapiens.csv

Record Sets and Fields:
- RecordSet @id: d18d681b-431c-4fc3-b535-1e26cb261034
    - Field @id: fd27a885-d6d0-40c9-b457-4521d36382af, name: %23+Peptides
    - Field @id: ef92de27-a908-49be-ad7e-06e91061083f, name: %23+PSMs
    - Field @id: 90a96fd9-9129-4231-ab45-fb5339970092, name: %23+Unique+Peptides
    - Field @id: be267551-8e7e-4785-bdcd-2712e3908b86, name: Abundances+%28Normalized%29%3A+F1%3A+Sample%2C+U%2C+1
    - Field @id: 1e69c639-904a-4d67-ab11-810a486a1575, name: Abundances+%28Normalized%29%3A+F2%3A+Sample%2C+U%2C+2
    - Field @id: 9c87cdf7-9708-4758-8799-b30a437a0cfa, name: Abundances+%28Normalized%29%3A+F3%3A+Sample%2C+U%2C+3
    - Field @id: bbf27463-a801-4082-8f76-f40d8ceaba05, name: Abundances+%28Normalized%29%3A+F4%3A+Sample%2C+T%2C+1
    - Field @id: a0ca02c5-83bb-40c9-80d4-e45765344211, name: Abundances+%28Normalized%29%3A+F5%3A+Sample%2C+T%2C+2
 

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Replace the list of `record_set_ids` with those you identified above.
Refer to each field or record set exclusively by their `@id` field.

In [4]:
# Identify record set @ids (example: use real ones from previous output)
# For this dataset, let's programmatically extract them (if any present)
record_sets_metadata = getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))
record_set_ids = [rset.id for rset in record_sets_metadata] if record_sets_metadata else []

dataframes = {}

for record_set_id in record_set_ids:
    # Load all records as a list of dicts
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        # mlcroissant returns text as bytes
        df = df.apply(lambda column: column.map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v))
        dataframes[record_set_id] = df
        print(f"DataFrame loaded for RecordSet {record_set_id}. Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"\nFirst 5 rows of {first_record_set_id}:")
    display(dataframes[first_record_set_id].head())
else:
    print("No RecordSets could be loaded into DataFrames.")

DataFrame loaded for RecordSet d18d681b-431c-4fc3-b535-1e26cb261034. Shape: (46395, 17)
Columns: ['fd27a885-d6d0-40c9-b457-4521d36382af', 'ef92de27-a908-49be-ad7e-06e91061083f', '90a96fd9-9129-4231-ab45-fb5339970092', 'be267551-8e7e-4785-bdcd-2712e3908b86', '1e69c639-904a-4d67-ab11-810a486a1575', '9c87cdf7-9708-4758-8799-b30a437a0cfa', 'bbf27463-a801-4082-8f76-f40d8ceaba05', 'a0ca02c5-83bb-40c9-80d4-e45765344211', 'ae9e2046-80ad-4543-a693-2177636ea9a8', '732a2571-23dc-4e6c-ae68-db6be94ab40c', 'cff3c9fb-56be-4f98-9ba9-d0fdbe8a34e1', 'e350e9c9-5761-4224-a377-88b987a8b36e', '86a47ac9-6bec-42af-a962-062a7fecd2e2', '6e725f01-0b71-4b39-91aa-bbf2285d9166', '5dccf6f8-5657-47a6-858a-5954665ce698', 'fea10532-2bb7-43dd-b10d-c4777e93f8f2', '4a4a275d-6c66-4b11-937d-e10a87efb065']
  fd27a885-d6d0-40c9-b457-4521d36382af ef92de27-a908-49be-ad7e-06e91061083f  \
0                                  224                                 3357   
1                        Modifications                          

,fd27a885-d6d0-40c9-b457-4521d36382af,ef92de27-a908-49be-ad7e-06e91061083f,90a96fd9-9129-4231-ab45-fb5339970092,be267551-8e7e-4785-bdcd-2712e3908b86,1e69c639-904a-4d67-ab11-810a486a1575,9c87cdf7-9708-4758-8799-b30a437a0cfa,bbf27463-a801-4082-8f76-f40d8ceaba05,a0ca02c5-83bb-40c9-80d4-e45765344211,ae9e2046-80ad-4543-a693-2177636ea9a8,732a2571-23dc-4e6c-ae68-db6be94ab40c,cff3c9fb-56be-4f98-9ba9-d0fdbe8a34e1,e350e9c9-5761-4224-a377-88b987a8b36e,86a47ac9-6bec-42af-a962-062a7fecd2e2,6e725f01-0b71-4b39-91aa-bbf2285d9166,5dccf6f8-5657-47a6-858a-5954665ce698,fea10532-2bb7-43dd-b10d-c4777e93f8f2,4a4a275d-6c66-4b11-937d-e10a87efb065
0,224,3357,224,12269096498.7745,13405731735.8719,15232519480.5129,11047975633.5699,11458901422.1352,13635619401.0208,Q09666,6.15,65,Neuroblast differentiation-associated protein ...,None,628.7,None,None
1,Modifications,# PSMs,Master Protein Accessions,"Abundances (Normalized): F1: Sample, U, 1","Abundances (Normalized): F2: Sample, U, 2","Abundances (Normalized): F3: Sample, U, 3","Abundances (Normalized): F4: Sample, T, 1","Abundances (Normalized): F5: Sample, T, 2","Abundances (Normalized): F6: Sample, T, 3",None,# Missed Cleavages,Annotated Sequence,Confidence,Sequence in Protein,Positions in Master Proteins,Positions in Proteins,Quan Usage
2,None,21,Q09666,2174227.34122055,1353331.1875,250279123.363814,588835.088431986,479330856.749587,120482237.827603,None,0,[K].ADLDVSGPK.[V],High,K.ADIDVSGPK.V; K.ADIDVSGPK.V; K.ADIDVSGPK.V; K...,Q09666 [1281-1289]; [2665-2673]; [2793-2801]; ...,[1281-1289]; [2665-2673]; [2793-2801]; [3049-3...,Used
3,None,43,Q09666,87724111.7210755,83036200,147278810.662231,76378060.7964661,107930096.324178,112465131.663754,None,0,[K].ADIDVSGPSVDTDAPDLDIEGPEGK.[L],High,K.ADIDVSGPSVDTDAPDLDIEGPEGK.L,Q09666 [1739-1763],[1739-1763],Used
4,None,20,Q09666,46610209.8581209,51144805.25,62842564.3306058,42448588.9580211,43106377.3645704,57719908.9636978,None,0,[K].ADIEISGPK.[V],High,K.ADIEISGPK.V,Q09666 [4343-4351],[4343-4351],Used


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes handling outliers, transforming numeric fields, or grouping data by key attributes.

**Note:** Make sure to reference all fields by their `@id`.

In [5]:
# For demonstration, select a numeric field from the first available record set.
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Attempt to auto-select a float or integer column by inspecting dtype or common numeric names
    numeric_field_id = None
    for col in df.columns:
        if any(key in col.lower() for key in ["count", "mw", "weight", "coverage", "abundance", "id"]):
            try:
                # If convertible to numeric, use it
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field_id = col
                    break
            except Exception:
                continue
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if numeric_field_id is None:
        print("No numeric field found in DataFrame for EDA.")
    else:
        print(f"Using numeric field '@id': {numeric_field_id}")

        # Filter records where the value > threshold (change as needed)
        threshold = df[numeric_field_id].quantile(0.9)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (example: any string/categorical field)
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No record sets with data available for EDA.")

No numeric field found in DataFrame for EDA.


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if record_set_ids and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize grouped mean if group_field is available
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,4))
        display_df = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
        sns.barplot(y=group_field, x=numeric_field_id, data=display_df)
        plt.title(f"Top 20 {group_field}s by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

No data available for visualization.


## 6. Conclusion
In this notebook, we loaded the FAIR² dataset containing mass spectrometry analysis of proteins from stimulated human mast cells, inspected the Croissant schema, loaded record sets by their `@id`, explored the available fields, filtered and normalized a numeric variable using `@id` references, and visualized distributions.

Further analysis can be performed by leveraging additional record sets or combining the data as required for your research question.
